# 🚀 FATFORMER-XLA: ĐÁNH GIÁ KIỂM CHUẨN NGOẠI VI COMMUNITYFORENSICS-EVAL (CVPR 2025)
### ĐO ĐẠC ĐA CHIỀU: 3 MÔ HÌNH × 2 ĐIỀU KIỆN (CLEAN & JPEG Q=30)

> **Người thực hiện**: Thành viên C *(Training & Evaluation Lead)*  
> **Mục tiêu**: Đánh giá năng lực tổng quát hóa Zero-shot và độ bền vững pháp y trên 2.000 ảnh Unseen của bộ kiểm chuẩn ngoại vi độc lập **CommunityForensics-Eval (CVPR 2025)**.  
> **3 Mô hình đối chứng**:  
> 1. **Baseline CVPR 2024** (Mô hình gốc không có SRM/Gating).  
> 2. **WiSE-FT (Mode 1)**: Trọng số nội suy đa miền cân bằng hiệu năng ảnh sạch.  
> 3. **Anchored Recovery (Mode 2)**: Trọng số phục hồi neo chặn chuyên biệt cho biến dạng nén sâu mạng xã hội.  
> **2 Điều kiện kiểm thử**:  
> - **Bảng A**: Ảnh sạch phòng lab (Clean).  
> - **Bảng B**: Ảnh nén sâu mạng xã hội (In-memory JPEG Quality=30).  
> **Hạ tầng kế thừa**: Checkpoint và dữ liệu `.tar` lưu trữ tại `MyDrive/Fatformer` (Google Drive 5TB).

### Bước 1: Kết nối Google Drive & Kiểm tra Tài nguyên Phần cứng GPU

In [ ]:
from google.colab import drive
import os, sys, torch

print("[*] Đang liên kết Google Drive 5TB...")
drive.mount('/content/drive')

print(f"[*] Phiên bản PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"[✓] Thiết bị tính toán: GPU {gpu_name} ({gpu_mem:.2f} GB VRAM)")
else:
    print("[!] CẢNH BÁO: Không tìm thấy GPU! Đang chạy trên CPU (tốc độ suy luận sẽ chậm hơn).")

!df -h /content | tail -1

### Bước 2: Đồng Bộ Mã Nguồn Mới Nhất Từ GitHub (Nhánh `hoang`)

In [ ]:
REPO_DIR = "/content/fatformer-xla"
REPO_URL = "https://github.com/LeeVHoangtk3/fatformer-xla.git"
BRANCH = "hoang"

if not os.path.exists(REPO_DIR):
    print(f"[*] Đang clone mã nguồn từ GitHub (nhánh {BRANCH}): {REPO_URL}...")
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    print(f"[*] Thư mục {REPO_DIR} đã tồn tại. Đang cập nhật commit mới nhất (git pull)...")
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}

%cd {REPO_DIR}
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"[✓] Thư mục làm việc hiện tại: {os.getcwd()}")

### Bước 3: Giải Nén Dữ Liệu `test_communityforensics.tar` Sang NVMe SSD Colab
Giải nén từ Google Drive sang SSD Colab `/content/dataset_local/` để đạt tốc độ nạp I/O cao nhất.

In [ ]:
import os, tarfile

DRIVE_TAR_PATH = "/content/drive/MyDrive/Fatformer/datasets/test_communityforensics.tar"
LOCAL_DIR = "/content/dataset_local/test_communityforensics"

# Kiểm tra nếu thư mục cục bộ đã có đủ 5 tập con thì không cần giải nén lại
need_extract = True
if os.path.exists(LOCAL_DIR):
    subsets = [d for d in os.listdir(LOCAL_DIR) if os.path.isdir(os.path.join(LOCAL_DIR, d))]
    if len(subsets) >= 5:
        print(f"[✓] Phát hiện dữ liệu 5 tập con đã có sẵn tại {LOCAL_DIR}. Bỏ qua bước giải nén!")
        need_extract = False

if need_extract:
    if not os.path.exists(DRIVE_TAR_PATH):
        raise FileNotFoundError(f"[!] Không tìm thấy tệp {DRIVE_TAR_PATH} trên Google Drive. Hãy chạy notebook chuẩn bị dữ liệu trước!")
    
    tar_sz_gb = os.path.getsize(DRIVE_TAR_PATH) / (1024 ** 3)
    print(f"[*] Đang giải nén {DRIVE_TAR_PATH} ({tar_sz_gb:.2f} GB) sang SSD Colab...")
    os.makedirs("/content/dataset_local", exist_ok=True)
    with tarfile.open(DRIVE_TAR_PATH, "r") as tar:
        tar.extractall("/content/dataset_local")
    print("[✓] Giải nén thành công toàn bộ 2.000 ảnh chuẩn hóa!")

# Kiểm tra số lượng ảnh
!find /content/dataset_local/test_communityforensics -type f -name "*.png" | wc -l

### Bước 4: Kích Hoạt Kiểm Chuẩn Tự Động 3 Mô Hình Trên 2 Điều Kiện
Đo đạc Macro ACC, Real ACC, Fake ACC, ROC-AUC và AP trên cả 5 họ mô hình bộ sinh.

In [ ]:
!python tools/eval_communityforensics.py \
    --data_dir /content/dataset_local/test_communityforensics \
    --output_dir /content/drive/MyDrive/Fatformer/logs/cfe_benchmark

### Bước 5: Xem Trực Quan Báo Cáo Nghiệm Thu Markdown
Hiển thị bảng ma trận Bảng A (Clean) và Bảng B (JPEG Q=30) trực tiếp trong Notebook.

In [ ]:
from IPython.display import display, Markdown

REPORT_MD = "/content/drive/MyDrive/Fatformer/logs/cfe_benchmark/communityforensics_benchmark_report.md"
if os.path.exists(REPORT_MD):
    with open(REPORT_MD, "r", encoding="utf-8") as f:
        display(Markdown(f.read()))
else:
    print(f"[*] Báo cáo sẽ xuất hiện tại: {REPORT_MD}")